# 05 — Explainable AI: Analysing the Agent's Noise Actions
**(Notebook 5 / 5 — requires `01_dataset_generation.ipynb` + `03_training.ipynb`)**

A trained RL agent has learned to assign noise parameters to every (moment, qubit) position of a circuit.  This notebook **decodes** those decisions and turns them into interpretable plots, giving insight into the noise the agent believes affects the quantum chip.

## What this notebook covers

| Section | What you learn |
|---------|----------------|
| 1 | Load a pre-trained agent and an evaluation dataset |
| 2 | Run `collect_actions` to harvest every noise action |
| 3 | Print a summary statistics table |
| 4 | Global noise distributions (histograms) |
| 5 | Noise breakdown per **gate type** (RX / RZ / CZ) |
| 6 | Noise breakdown per **qubit** |
| 7 | Noise as a function of **circuit depth** (spatial profile) |
| 8 | Per-qubit spatial profile for a chosen noise channel |
| 9 | Pairwise **correlation** between noise channels |
| 10 | Bar chart: mean noise per gate type |
| 11 | Comparison with the ground-truth noise parameters |

**Outputs written by this notebook:** `results/analysis/3q/`

## 1. Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from rlnoise.config import (
    DatasetConfig, NoiseConfig, GateSpecificNoise,
    GymEnvConfig, RewardConfig, AgentConfig,
)
from rlnoise.dataset import DatasetGenerator, CircuitDataset
from rlnoise.circuit_encoder import CircuitEncoder
from rlnoise.gym_env import QuantumCircuitEnv
from rlnoise.rl_agent import RLAgent

from rlnoise.analysis import (
    collect_actions,
    noise_summary,
    plot_noise_distributions,
    plot_noise_by_gate,
    plot_noise_by_qubit,
    plot_spatial_noise,
    plot_spatial_noise_per_qubit,
    plot_noise_correlation,
    plot_mean_noise_per_gate,
    NOISE_CHANNELS, NOISE_LABELS,
)

RESULTS_DIR = Path("results/analysis/3q")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(0)

## 2. Load the Pre-trained Agent

Adjust the paths below to point at your own experiment.

In [ ]:
# ── Noise model used during training ─────────────────────────────────────────
noise_config = NoiseConfig(
    noise_list=[
        GateSpecificNoise(gate="rz", noise_channel="depolarizing",  noise_parameter=0.02),
        GateSpecificNoise(gate="cz", noise_channel="depolarizing",  noise_parameter=0.02),
        GateSpecificNoise(gate="rx", noise_channel="damping",        noise_parameter=0.03),
        GateSpecificNoise(gate="cz", noise_channel="damping",        noise_parameter=0.03),
        GateSpecificNoise(gate="rx", noise_channel="coherent_x",     noise_parameter=0.04),
        GateSpecificNoise(gate="rz", noise_channel="coherent_z",     noise_parameter=0.03),
    ]
)

# ── Dataset used during training ──────────────────────────────────────────────
DATASET_PATH = Path("datasets/dataset_3q.npz")
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at '{DATASET_PATH}'.\n"
        "Please run '01_dataset_generation.ipynb' (Section 4) first."
    )

dataset = CircuitDataset.load(str(DATASET_PATH))
print(f"Loaded dataset from '{DATASET_PATH}'")
print(dataset)

In [ ]:
# ── Environment & agent configuration (must match training) ───────────────────
encoder = CircuitEncoder(primitive_gates=["rx", "rz", "cz"])

env_config = GymEnvConfig(
    kernel_size=3,
    action_space_max_value=0.06,
    val_split=0.2,
    enable_only_depolarizing=False,
)

reward_config = RewardConfig(metric="trace", function="inverted_squared", alpha=20.0)

agent_config = AgentConfig(
    policy="MlpPolicy",
    features_dim=64, filter_size=1, n_filters=32,
    pi_net_arch=[32, 32], vf_net_arch=[32, 32],
    n_steps=2000, batch_size=400, learning_rate=1e-4,
)

env = QuantumCircuitEnv(
    dataset=dataset,
    encoder=encoder,
    env_config=env_config,
    reward_config=reward_config,
)

# ── Load saved model weights ──────────────────────────────────────────────────
AGENT_PATH = "agents/3q/model"
if not Path(AGENT_PATH + ".zip").exists():
    raise FileNotFoundError(
        f"Trained agent not found at '{AGENT_PATH}.zip'.\n"
        "Please run '03_training.ipynb' (Section 5) first."
    )

agent = RLAgent(
    env=env,
    agent_config=agent_config,
    model_path=AGENT_PATH,
)
print(f"Agent loaded from '{AGENT_PATH}'")
print(agent)

## 3. Collect Agent Actions

`collect_actions` runs the agent **deterministically** over every circuit in the array and records the four scaled noise values it writes into each (moment, qubit) slot.

Adjust `N_EVAL_CIRCUITS` to trade off speed vs. statistical resolution.

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
N_EVAL_CIRCUITS = 200   # circuits to analyse
EVAL_DEPTH      = 15    # moments per circuit
EVAL_CLIFFORD   = False # non-Clifford = harder generalisation test

# ── Generate a fresh evaluation dataset ──────────────────────────────────────
eval_cfg = DatasetConfig(
    n_circuits=N_EVAL_CIRCUITS,
    moments=EVAL_DEPTH,
    qubits=3,
    primitive_gates=["rz", "rx", "cz"],
    clifford=EVAL_CLIFFORD,
    mixed=False,
)
eval_dataset = DatasetGenerator(eval_cfg, noise_config).generate(verbose=False)
print(f"Evaluation dataset: {eval_dataset}")

# ── Collect actions ───────────────────────────────────────────────────────────
actions_data = collect_actions(agent, eval_dataset.circuits, verbose=True)

print(f"\nResult shapes:")
for ch in NOISE_CHANNELS:
    print(f"  {ch:12s}: {actions_data[ch].shape}")
print(f"  gate_type   : {actions_data['gate_type'].shape}")

## 4. Summary Statistics

`noise_summary` prints mean, std, min and max for every noise channel across all gate positions.

Two filters are available (both default to `True`):
- **`skip_identity`** — exclude positions where no gate was applied (gate_type `"id"`).
- **`skip_zero`** — exclude positions where the agent output **exactly zero** for that channel.  With `skip_zero=True` the statistics reflect only positions where the agent actively applied noise, which is usually more informative than including the many "I chose not to act" zeros.

In [ ]:
# Default: identity positions AND zero-output positions excluded
print("── skip_identity=True, skip_zero=True (default) ──")
print(noise_summary(actions_data, skip_identity=True, skip_zero=True))

print()

# Compare: include zero-output positions to see how many "no-action" slots exist
print("── skip_identity=True, skip_zero=False ──")
print(noise_summary(actions_data, skip_identity=True, skip_zero=False))

## 5. Global Noise Distributions

One histogram per noise channel, aggregated over all circuits, moments and qubits.  The dashed line marks the mean; the shaded band shows ±1σ.

Compare the means to the ground-truth noise parameters in `noise_config` to see how close the agent gets.

In [ ]:
fig = plot_noise_distributions(
    actions_data,
    skip_identity=True,
    skip_zero=True,        # exclude positions where agent output zero
    bins=40,
    filepath=str(RESULTS_DIR / "noise_distributions.png"),
)
plt.show()

## 6. Noise Breakdown by Gate Type

Does the agent apply different amounts of noise after an RX gate vs. an RZ gate vs. a CZ gate?  Rows = noise channel, columns = gate type.

In [ ]:
fig = plot_noise_by_gate(
    actions_data,
    skip_zero=True,
    bins=30,
    filepath=str(RESULTS_DIR / "noise_by_gate.png"),
)
plt.show()

The compact bar-chart version of the same information — easier to compare means across gate types at a glance:

In [ ]:
fig = plot_mean_noise_per_gate(
    actions_data,
    skip_zero=True,
    filepath=str(RESULTS_DIR / "mean_noise_per_gate.png"),
)
plt.show()

## 7. Noise Breakdown by Qubit

Are some qubits noisier than others?  If the underlying hardware has qubit-specific noise, a well-trained agent should assign higher noise values to the noisier qubits.

In [ ]:
fig = plot_noise_by_qubit(
    actions_data,
    skip_identity=True,
    skip_zero=True,
    bins=30,
    filepath=str(RESULTS_DIR / "noise_by_qubit.png"),
)
plt.show()

## 8. Spatial Noise Profile — Noise vs. Circuit Depth

Does the agent apply more noise at the start or the end of a circuit?  A flat profile suggests the agent has learned a *Markovian* (position-independent) noise model, while a sloped profile indicates non-Markovian or time-dependent noise.

In [ ]:
fig = plot_spatial_noise(
    actions_data,
    skip_identity=True,
    skip_zero=True,
    filepath=str(RESULTS_DIR / "spatial_noise.png"),
)
plt.show()

### 8.1 Per-qubit spatial profile

Select a single noise channel and overlay all qubits on one plot:

In [ ]:
for channel in NOISE_CHANNELS:
    fig = plot_spatial_noise_per_qubit(
        actions_data,
        channel=channel,
        skip_identity=True,
        skip_zero=True,
        filepath=str(RESULTS_DIR / f"spatial_noise_per_qubit_{channel}.png"),
    )
    plt.show()

## 9. Pairwise Noise Correlations

Are noise channels applied independently or do they co-vary?  A high correlation between e.g. damping and depolarising would suggest the agent treats them as coupled — which may indicate an entangled noise mechanism in the underlying hardware.

- **Diagonal**: histogram of each channel
- **Off-diagonal**: scatter plot of two channels; Pearson *r* in title

In [ ]:
fig = plot_noise_correlation(
    actions_data,
    skip_identity=True,
    skip_zero=True,        # positions silent on ALL channels are excluded
    max_points=3000,
    filepath=str(RESULTS_DIR / "noise_correlation.png"),
)
plt.show()

## 10. Comparison with Ground-Truth Parameters

We know the exact noise parameters used to generate the dataset.  Let's compare the agent's mean action per gate type against the expected ground-truth value.

The table below extracts the mean noise the agent assigned to each gate type for each channel and compares it to the simulation value.

In [ ]:
from rlnoise.analysis import _flat_values, NOISE_LABELS

# Ground-truth parameters used in noise_config (indexed by channel → gate)
# Manually extracted from noise_config above for comparison
ground_truth = {
    # channel       gate   value
    "depol":      {"rz": 0.02, "cz": 0.02},
    "reset":      {"rx": 0.03, "cz": 0.03},
    "epsilon_x":  {"rx": 0.04},
    "epsilon_z":  {"rz": 0.03},
}

print(f"{'Channel':<22} {'Gate':<6} {'Agent mean':>12} {'Std':>10} {'Ground truth':>14} {'Error':>10}")
print("-" * 78)

for ch, gate_dict in ground_truth.items():
    for gate, gt_val in gate_dict.items():
        vals = _flat_values(actions_data, ch, gate_filter=gate,
                            skip_identity=True, skip_zero=True)
        if vals.size == 0:
            continue
        mean, std = vals.mean(), vals.std()
        error = mean - gt_val
        sign = "+" if error >= 0 else ""
        print(f"{NOISE_LABELS[ch]:<22} {gate.upper():<6} {mean:>12.5f} "
              f"{std:>10.5f} {gt_val:>14.5f} {sign}{error:>9.5f}")

In [ ]:
# Visual comparison: mean agent noise vs. ground truth, per channel and gate
fig, axes = plt.subplots(1, len(NOISE_CHANNELS), figsize=(14, 4))

from rlnoise.analysis import GATE_COLORS, NOISE_COLORS

for ax, ch in zip(axes, NOISE_CHANNELS):
    present = sorted({g for g in ("rx", "rz", "cz")
                      if np.any(actions_data["gate_type"] == g)})
    x = np.arange(len(present))
    means = []
    stds  = []
    gts   = []
    for gate in present:
        vals = _flat_values(actions_data, ch, gate_filter=gate,
                            skip_identity=True, skip_zero=True)
        means.append(vals.mean() if vals.size > 0 else 0.0)
        stds.append(vals.std()   if vals.size > 0 else 0.0)
        gts.append(ground_truth.get(ch, {}).get(gate, np.nan))

    colors = [GATE_COLORS[g] for g in present]
    ax.bar(x - 0.18, means, width=0.35, yerr=stds, capsize=5,
           color=colors, edgecolor="white", label="Agent")
    ax.scatter(x + 0.18, gts, marker="D", s=60, color="black",
               zorder=5, label="Ground truth")
    ax.set_xticks(x)
    ax.set_xticklabels([g.upper() for g in present])
    ax.set_title(NOISE_LABELS[ch], fontsize=9)
    ax.set_ylabel("Noise value")
    ax.legend(fontsize=8)
    ax.grid(True, axis="y", alpha=0.3)

fig.suptitle("Agent mean noise vs. ground truth (bars = agent ± std, diamonds = GT)",
             fontsize=12)
fig.tight_layout()
fig.savefig(str(RESULTS_DIR / "agent_vs_ground_truth.png"), dpi=150)
plt.show()

## 11. Working with the Raw Action Arrays

All collected data is plain NumPy arrays, making it easy to do custom analysis.  Below are a few quick examples.

In [ ]:
# ── 1.  Per-circuit mean depolarising parameter ───────────────────────────────
# Shape (n_circuits,) — one value per circuit
depol_per_circuit = actions_data["depol"].mean(axis=(1, 2))   # mean over moments & qubits

plt.figure(figsize=(8, 3))
plt.plot(depol_per_circuit, color="#9b19f5", lw=1)
plt.axhline(depol_per_circuit.mean(), color="black", linestyle="--",
            label=f"Mean = {depol_per_circuit.mean():.4f}")
plt.xlabel("Circuit index")
plt.ylabel("Mean λ (depolarising)")
plt.title("Per-circuit mean depolarising parameter")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "depol_per_circuit.png"), dpi=150)
plt.show()

In [ ]:
# ── 2.  Noise map: moment × qubit heatmap (averaged over circuits) ────────────
fig, axes = plt.subplots(1, len(NOISE_CHANNELS), figsize=(14, 4))

for ax, ch in zip(axes, NOISE_CHANNELS):
    # Mean over circuits → shape (n_moments, n_qubits)
    heatmap = actions_data[ch].mean(axis=0)
    im = ax.imshow(heatmap.T, aspect="auto", cmap="viridis", origin="lower")
    ax.set_xlabel("Moment")
    ax.set_ylabel("Qubit")
    ax.set_yticks(range(actions_data["n_qubits"]))
    ax.set_yticklabels([f"q{q}" for q in range(actions_data["n_qubits"])])
    ax.set_title(NOISE_LABELS[ch], fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle("Noise heatmap: mean over circuits  (moment × qubit)", fontsize=12)
fig.tight_layout()
fig.savefig(str(RESULTS_DIR / "noise_heatmap.png"), dpi=150)
plt.show()

In [ ]:
# ── 3.  Fraction of positions where each channel is near zero (<1e-4) ─────────
#   A channel that is almost always zero may not be relevant for this chip.
mask_nonid = actions_data["gate_type"] != "id"

print(f"{'Channel':<22} {'Near-zero %':>12}  (threshold = 1e-4, identity excluded)")
print("-" * 50)
for ch in NOISE_CHANNELS:
    vals = actions_data[ch][mask_nonid]
    pct = 100.0 * (vals < 1e-4).sum() / vals.size
    print(f"{NOISE_LABELS[ch]:<22} {pct:>11.1f} %")